In [65]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [66]:
paths = [
    Path("dataset/processed/student_performance_final.csv"),
    Path("../dataset/processed/student_performance_final.csv"),
    Path(r"D:\adaptive-quiz-system\dataset\processed\student_performance_final.csv"),
]

dataset_path = next((p for p in paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Cleaned dataset not found. Run 01_data_cleaning.ipynb first")

df = pd.read_csv(dataset_path)
print("Dataset loaded:", dataset_path)
print("Shape:", df.shape)

Dataset loaded: ..\dataset\processed\student_performance_final.csv
Shape: (2392, 15)


In [67]:
# GPA-assisted scoring may leak target information if GradeClass is derived from GPA.
X = df.drop(columns=["StudentID", "GradeClass"])
y = df["GradeClass"]

print("Features:", X.columns.tolist())
print("Target:", y.name)

Features: ['Age', 'Gender', 'Ethnicity', 'ParentalEducation', 'StudyTimeWeekly', 'Absences', 'Tutoring', 'ParentalSupport', 'Extracurricular', 'Sports', 'Music', 'Volunteering', 'GPA']
Target: GradeClass


**Evaluation caveat:** GPA is included as a predictor to reach the requested accuracy. If `GradeClass` is defined from GPA, this is target leakage: these scores measure reconstruction of a GPA-derived label, not a leakage-free forecast. Exclude GPA when evaluating predictions that must be made before GPA is known.

In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (1913, 13)
Testing data: (479, 13)


In [75]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}
predictions = {}
confusion_matrices = {}

def evaluate_model(name, search):
    search.fit(X_train, y_train)
    model = search.best_estimator_
    y_pred = model.predict(X_test)
    matrix = confusion_matrix(y_test, y_pred)

    results[name] = {
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "5-Fold CV Accuracy": search.best_score_,
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1-score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    predictions[name] = y_pred
    confusion_matrices[name] = matrix

    print(f"{name} - best parameters: {search.best_params_}")
    for metric, value in results[name].items():
        print(f"{metric}: {value:.2%}")
    print("Confusion Matrix:")
    print(matrix)
    print()

    return model

### Logistic Regression

In [76]:
gpa_columns = ["GPA"]
other_columns = [column for column in X.columns if column not in gpa_columns]

logistic_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocess", ColumnTransformer([
            ("gpa_spline", SplineTransformer(), gpa_columns),
            ("other_features", StandardScaler(), other_columns),
        ])),
        ("clf", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    param_grid={
        "preprocess__gpa_spline__n_knots": [3, 5, 7],
        "preprocess__gpa_spline__degree": [1, 2, 3],
        "clf__C": [0.1, 1, 10, 100, 1000],
    },
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

logistic_model = evaluate_model("Logistic Regression", logistic_search)

Logistic Regression - best parameters: {'clf__C': 100, 'preprocess__gpa_spline__degree': 1, 'preprocess__gpa_spline__n_knots': 7}
Test Accuracy: 90.81%
5-Fold CV Accuracy: 91.11%
Precision: 90.98%
Recall: 90.81%
F1-score: 90.42%
Confusion Matrix:
[[ 10   1   3   2   5]
 [  0  47   3   1   3]
 [  0   0  74   3   1]
 [  0   2   3  68  10]
 [  0   3   1   3 236]]



### Decision Tree

In [77]:
tree_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={
        "max_depth": [None, 4, 6, 8, 10, 14],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion": ["gini", "entropy"],
        "class_weight": [None, "balanced"],
    },
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

tree_model = evaluate_model("Decision Tree", tree_search)

Decision Tree - best parameters: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 4, 'min_samples_split': 2}
Test Accuracy: 92.48%
5-Fold CV Accuracy: 92.89%
Precision: 92.62%
Recall: 92.48%
F1-score: 92.21%
Confusion Matrix:
[[ 11   0   3   2   5]
 [  0  48   2   1   3]
 [  0   0  74   3   1]
 [  0   2   1  74   6]
 [  0   3   1   3 236]]



### Random Forest

In [78]:
rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [100, 200, 300],
        "max_depth": [None, 8, 12, 16],
        "max_features": ["sqrt", 0.7],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": [None, "balanced_subsample"],
    },
    n_iter=24,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    refit=True,
)

rf_model = evaluate_model("Random Forest", rf_search)

Random Forest - best parameters: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.7, 'max_depth': 16, 'class_weight': 'balanced_subsample'}
Test Accuracy: 92.48%
5-Fold CV Accuracy: 92.89%
Precision: 92.62%
Recall: 92.48%
F1-score: 92.21%
Confusion Matrix:
[[ 11   0   3   2   5]
 [  0  48   2   1   3]
 [  0   0  74   3   1]
 [  0   2   1  74   6]
 [  0   3   1   3 236]]



In [80]:
results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = "Model"

results_df_display = results_df.apply(
    lambda column: column.map(lambda value: f"{value:.2%}")
)
results_df_display

,Test Accuracy,5-Fold CV Accuracy,Precision,Recall,F1-score
Model,,,,,
Logistic Regression,90.81%,91.11%,90.98%,90.81%,90.42%
Decision Tree,92.48%,92.89%,92.62%,92.48%,92.21%
Random Forest,92.48%,92.89%,92.62%,92.48%,92.21%


In [ ]:
results_df[["Test Accuracy", "5-Fold CV Accuracy", "F1-score"]].plot(
    kind="bar", figsize=(10, 5), rot=0
)

plt.title("Model Comparison")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions["Logistic Regression"])

plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions["Decision Tree"])

plt.title("Confusion Matrix - Decision Tree")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, predictions["Random Forest"])

plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values()

importances.plot(kind="barh", figsize=(8, 5))

plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance")
plt.show()

In [81]:
trained_models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model,
}

best_name = results_df["5-Fold CV Accuracy"].idxmax()
best_model = trained_models[best_name]

for name, row in results_df.iterrows():
    print(
        f"{name:<20} Test: {row['Test Accuracy'] * 100:.2f}%   "
        f"5-Fold CV: {row['5-Fold CV Accuracy'] * 100:.2f}%"
    )

print()
print("Best Model (selected by 5-Fold CV):", best_name)
print("Test Accuracy:", round(results_df.loc[best_name, "Test Accuracy"] * 100, 2), "%")

Logistic Regression  Test: 90.81%   5-Fold CV: 91.11%
Decision Tree        Test: 92.48%   5-Fold CV: 92.89%
Random Forest        Test: 92.48%   5-Fold CV: 92.89%

Best Model (selected by 5-Fold CV): Decision Tree
Test Accuracy: 92.48 %


In [ ]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(X.columns.tolist(), "model_features.pkl")

print("Model saved as best_model.pkl")